# Leakage-safe crux probe: fit on train/validation, evaluate on the untouched test period

The published crux probe fits and evaluates inside a random 60/40 split of the test partition, which the
manuscript flags as potentially optimistic in absolute terms because temporally adjacent flows can land on
both sides. This notebook runs the stricter protocol promised in the limitations: one-vs-rest probes are
fitted on representations from the training and validation partitions only and evaluated once on the
untouched provenance-ordered test partition. If the M0-versus-prune80 AUC contrast holds here, the crux
conclusion no longer depends on the within-test split at all.

Probe setup is otherwise identical to `explain.crux_probe`: `LogisticRegression(max_iter=500, C=1.0,
class_weight="balanced")`, one-vs-rest per class. Run on a GPU runtime for the feature extraction.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch
from src.config import CFG, PATHS, set_all_seeds
from src import data as D, models as M, compression as C, explain as EXP, train as TR

SEED = CFG['anchor_seed']
set_all_seeds(SEED)
ARCH = 'cnn1d'; DATASET = 'ciciot2023'
print('anchor seed:', SEED)

In [ ]:
import torch
print('device:', 'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU (SLOW)')
assert torch.cuda.is_available(), \
    'CPU runtime detected. Switch to a T4 GPU (Runtime -> Change runtime type) before running.'

In [ ]:
# Load the anchor (M0), rebuild the primary split, and reproduce the prune80 model
df = D.clean(D.load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = D.temporal_within_capture_split(df, seed=SEED)
feat_cols = TR.feature_columns(df)

from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder().fit(df['label'].to_numpy())
scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
n_classes = len(le.classes_)

ck = torch.load(PATHS.model(DATASET, ARCH, 'M0', SEED), map_location='cpu', weights_only=False)
sd = ck['state_dict']
ch = (int(sd['conv.0.weight'].shape[0]), int(sd['conv.3.weight'].shape[0]))
anchor = M.build(ARCH, len(feat_cols), n_classes, channels=ch)
anchor.load_state_dict(sd); anchor = anchor.to(C.DEVICE).eval()

set_all_seeds(SEED)
model_p80, _le, _sc = C.prune_and_finetune(anchor, df, DATASET, splits, SEED, 0.80, arch=ARCH, verbose=False)
model_p80 = model_p80.to(C.DEVICE).eval()
print('anchor + prune80 ready')

In [ ]:
# Extract frozen penultimate features for train, validation, and test partitions, both models.
fit_parts = {}
for which in ['train', 'val']:
    f0, _, yw = EXP.extract_features(anchor, df, splits, scaler, feat_cols, le, which=which)
    fc, _, _  = EXP.extract_features(model_p80, df, splits, scaler, feat_cols, le, which=which)
    fit_parts[which] = (f0, fc, yw)
    print(which, f0.shape)

f0_te, _, y_te = EXP.extract_features(anchor, df, splits, scaler, feat_cols, le, which='test')
fc_te, _, _    = EXP.extract_features(model_p80, df, splits, scaler, feat_cols, le, which='test')
print('test', f0_te.shape)

In [ ]:
# Fit pool: train+val, capped per class at 8,000 rows (rare classes kept whole).
rng = np.random.default_rng(SEED)
f0_fit = np.concatenate([fit_parts['train'][0], fit_parts['val'][0]])
fc_fit = np.concatenate([fit_parts['train'][1], fit_parts['val'][1]])
y_fit  = np.concatenate([fit_parts['train'][2], fit_parts['val'][2]])

keep = []
for c in np.unique(y_fit):
    idx = np.where(y_fit == c)[0]
    if len(idx) > 8000:
        idx = rng.choice(idx, 8000, replace=False)
    keep.append(idx)
keep = np.concatenate(keep)
f0_fit, fc_fit, y_fit = f0_fit[keep], fc_fit[keep], y_fit[keep]
print('fit pool:', f0_fit.shape, '| eval (untouched test):', f0_te.shape)

In [ ]:
# The 13 measurable classes and the 10 that collapse, matching the manuscript.
UNSTABLE = {'DoS-TCP_Flood', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-SYN_Flood'}
nbnd = pd.read_csv(PATHS.tables('baseline', 'cnn1d_M0_null_band_5seed.csv')).set_index('label')
measurable = [c for c in nbnd.index[nbnd['tier'] == 'measurable'] if c not in UNSTABLE]
assert len(measurable) == 13
mat = pd.read_csv(PATHS.tables('compression', 'cnn1d_per_class_recall_matrix.csv'), index_col=0)
collapsed10 = [c for c in measurable
               if (nbnd.loc[c, 'mean'] - mat.loc[c, 'prune80']) > nbnd.loc[c, 'null_band_2sigma']]
print(f'{len(collapsed10)} collapsed measurable classes')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def probe_auc(f_fit, f_eval, c):
    yb_fit = (y_fit == c).astype(int)
    yb_te  = (y_te == c).astype(int)
    if yb_fit.sum() < 5 or yb_te.sum() < 2:
        return np.nan
    clf = LogisticRegression(max_iter=500, C=1.0, class_weight='balanced')
    clf.fit(f_fit, yb_fit)
    return roc_auc_score(yb_te, clf.predict_proba(f_eval)[:, 1])

rows = []
for c_name in measurable:
    c = int(np.where(le.classes_ == c_name)[0][0])
    a0 = probe_auc(f0_fit, f0_te, c)
    ac = probe_auc(fc_fit, fc_te, c)
    rows.append({'label': c_name, 'collapsed': c_name in collapsed10,
                 'auc_M0': round(a0, 4), 'auc_prune80': round(ac, 4),
                 'auc_drop': round(a0 - ac, 4)})
    print(f"  {c_name:22s} M0 {a0:.4f}  p80 {ac:.4f}  drop {a0-ac:+.4f}"
          f"  {'(collapsed)' if c_name in collapsed10 else ''}")

safe = pd.DataFrame(rows)

In [ ]:
# Summary against the published within-test protocol (mean drop 0.009, all collapsed-class AUC > 0.91)
coll = safe[safe['collapsed']]
print(f'LEAKAGE-SAFE protocol, {len(coll)} collapsed classes:')
print(f'  mean AUC drop: {coll["auc_drop"].mean():+.4f}')
print(f'  max |drop|:    {coll["auc_drop"].abs().max():.4f}')
print(f'  min prune80 AUC: {coll["auc_prune80"].min():.4f}')
print()
print('published within-test protocol: mean drop 0.009, all collapsed-class prune80 AUC > 0.91')
print()
print(safe.to_string(index=False))

out = PATHS.tables('explain', 'crux_probe_leakage_safe.csv')
safe.to_csv(out, index=False)
print()
print('saved:', out)

## Outputs

`crux_probe_leakage_safe.csv` reports, for all 13 measurable classes, one-vs-rest probe AUC on M0 and
prune80 where the probe is fitted on train/validation representations and evaluated once on the untouched
provenance-ordered test partition. The quantities of interest are the mean AUC drop over the ten collapsed
classes and the minimum prune80 AUC, compared with the published within-test values.

In [ ]:
# --- End-of-unit discipline: commit + push (credentials restored from Drive) ---
import subprocess, shutil
DRIVE_ROOT = '/content/drive/MyDrive/IoT_Trust_Research'
for f in ['.gitconfig', '.git-credentials']:
    src = os.path.join(DRIVE_ROOT, f)
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
subprocess.run(['git', 'add', '-A'], check=True)
print(subprocess.run(['git', 'commit', '-m',
    'leakage-safe crux probe: train/val-fitted, untouched temporal test evaluation'],
    capture_output=True, text=True).stdout)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')

In [ ]:
# --- Commit + push (identity set explicitly, then persisted to Drive) ---
import os, shutil, subprocess
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
DRIVE_ROOT = '/content/drive/MyDrive/IoT_Trust_Research'
os.chdir(REPO)

# identity: set explicitly, no dependence on Drive files
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
cred = os.path.join(DRIVE_ROOT, '.git-credentials')
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials')

subprocess.run(['git', 'add', '-A'], check=True)
r = subprocess.run(['git', 'commit', '-m',
    'leakage-safe crux probe + head-vs-body ablation: notebooks and results'],
    capture_output=True, text=True)
print(r.stdout or r.stderr)
p = subprocess.run(['git', 'push'], capture_output=True, text=True)
print(p.stderr or p.stdout or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)

# persist identity to Drive root so future sessions restore correctly
shutil.copy('/root/.gitconfig', os.path.join(DRIVE_ROOT, '.gitconfig'))
print('.gitconfig saved to Drive root')